# Policy curve

Builds directly on the `conversion` decile table (notebook 03) for the DR-learner. Question: if you can only afford to advertise to a fraction of users, which fraction, and how much value do you capture?

For each targeting fraction (top decile, top 2 deciles, ...), compute:
- **cumulative incremental conversions** = sum of `actual_ate x n` across the included deciles (rate x headcount = an actual count, not a rate)
- **cost** = `cost_per_impression x people targeted`
- **benefit** = `value_per_conversion x cumulative incremental conversions`
- **net profit** = `benefit - cost`

Assumed economics (this dataset has no real business numbers - non-uniformly subsampled, per notebook 01/02 caveats): **$0.01/impression** (~$10 CPM, typical display/retargeting), **$50/conversion** (illustrative average order value). Flagged explicitly as assumptions, not real Criteo figures.

In [ ]:
import sys
sys.path.insert(0, "../src")

from uplift.data import load_sample, split_train_eval
from uplift.cate_models import fit_dr_learner, predict_dr_learner
from uplift.evaluation import decile_table
from uplift.policy import policy_curve
import pandas as pd

df = load_sample()
train, eval_ = split_train_eval(df)
outcome = "conversion"

dr_model = fit_dr_learner(train, outcome)
dr_cate = predict_dr_learner(dr_model, eval_)

tbl = decile_table(eval_, dr_cate, outcome)
naive_curve = policy_curve(tbl)
naive_curve

## The naive curve says "target 100%" — don't trust it yet

Net profit keeps climbing all the way to `fraction_targeted=1.0`. But recall from notebook 03: most middle deciles had confidence intervals that **crossed zero** — statistically indistinguishable from no effect. The naive curve sums up those noisy point estimates as if they were solid numbers.

**Why the curve looks profitable anyway**: breakeven rate = `cost_per_impression / value_per_conversion = 0.01/50 = 0.0002`. That's 5-15x *smaller* than the confidence interval half-widths seen in the middle deciles (~0.001-0.003). The bar for "looking profitable" is so low that a decile with a truly-zero effect has a good chance of clearing it from noise alone. The monotonic climb is largely an artifact of lenient assumed economics relative to what the statistics can actually support — not evidence that targeting everyone is genuinely profitable.

## Statistically honest version: Bonferroni-corrected, zero out non-significant deciles

Two fixes layered on top of the naive curve:

1. **Multiple comparisons**: testing 10 deciles at once means ~0.5 false positives expected per model just from a per-test 5% error rate, even under a true null. To control the *overall* false-positive rate across all 10 simultaneous tests, use a **Bonferroni correction**: `alpha = 0.05 / 10 = 0.005` (99.5% CIs) instead of the default 95%.
2. **Zero out non-significant deciles**: any decile whose (corrected) CI still includes zero gets `actual_ate` set to 0 before computing cumulative incremental conversions — don't credit a decile with conversions it can't statistically support.

In [ ]:
from uplift.policy import conservative_decile_table

n_bins = 10
tbl_bonferroni = decile_table(eval_, dr_cate, outcome, n_bins=n_bins, alpha=0.05 / n_bins)
print(tbl_bonferroni[["decile", "n", "actual_ate", "ci_low", "ci_high"]].to_string(index=False))

honest_curve = policy_curve(conservative_decile_table(tbl_bonferroni))
honest_curve

## Conclusion

The Bonferroni-honest optimal cutoff comes out at **fraction_targeted=0.7** (net profit ≈13,509), down from the naive "target 100%" (≈16,310). Decile 9 (top 10%) is overwhelmingly the dominant, clearly-significant contributor (`actual_ate=0.0078`, CI `0.0023` to `0.0134` even after correction). Deciles 3, 4, and 6 barely survive the corrected threshold (`ci_low` just above zero) — real but marginal.

**Two-tier recommendation, not a single number:**
- **High confidence: target the top decile (~10% of users).** This is the one segment with a dramatic, robustly-significant effect across every CATE model tried (S/T/X/DR-learner, causal forest) and both outcomes (`visit`, `conversion`).
- **Defensible but marginal: expanding to ~70%** captures some additional real value (deciles 3/4/6), but rests on effects that only barely survive a strict multiple-comparisons correction — worth flagging as lower-confidence in the write-up rather than presenting as equally solid.

**Caveat for the write-up**: this whole exercise depends on assumed economics ($0.01/impression, $50/conversion) that aren't Criteo's real numbers — the *optimal fraction* would shift if those assumptions changed, though the qualitative finding (top decile dominates, rest is marginal/noisy) is robust to reasonable changes in those assumptions.

## Split-sample check: select deciles on one half, re-measure on the other

The Bonferroni curve above picks significant deciles and credits them using the *same* data, so the kept estimates are inflated (winner's curse). `select_then_remeasure` splits the eval set in half: half A chooses which deciles pass the corrected test, half B (independent noise) supplies the effect credited to them. Repeated over several seeds to see how stable the selection is.

In [ ]:
from uplift.policy import select_then_remeasure

for seed in range(5):
    sel, meas, remeasured = select_then_remeasure(eval_, dr_cate, outcome, seed=seed)
    picked = sel.loc[(sel["ci_low"] > 0) | (sel["ci_high"] < 0), "decile"].tolist()
    curve = policy_curve(remeasured)
    best = curve.loc[curve["net_profit"].idxmax()]

    print(f"seed {seed}: deciles selected in half A = {picked}")
    print("   half A actual_ate:", sel.set_index("decile").loc[picked, "actual_ate"].round(4).to_dict())
    print("   half B actual_ate:", meas.set_index("decile").loc[picked, "actual_ate"].round(4).to_dict())
    print(f"   optimal fraction = {best['fraction_targeted']:.1f}, net profit = {best['net_profit']:.0f}")